# Análisis Topológico de la Ingesta de Ácido Fólico en Mujeres Chilenas
## Semanas 1 y 2 — Comprensión de Datos, EDA, Clustering Clásico y TDA Mapper

---

**Equipo 5** | Mariel Álvarez · Viviana · Ana · Álvaro · Jorge  
**Curso:** Uso de Geometría y Topología para la Ciencia de Datos  
**Fecha:** Mayo – Junio 2026  

---

### Referencia al Plan de Trabajo

Este notebook implementa las tareas de las **Semanas 1 y 2** del plan de trabajo del Equipo 5:

| Semana | Tareas cubiertas |
|--------|------------------|
| **Semana 1** | T1.1 Revisión de diccionarios y variables · T1.2 EDA · T1.3 Estrategia de aislamiento de covariables · T1.4 Diseño de pipeline |
| **Semana 2** | T2.1 Preprocesamiento para TDA · T2.2 Clustering clásico · T2.3 Mapper lente UMAP-AF · T2.4 Mapper lente salud neonatal · T2.5 Interpretación · T2.6 Homología persistente |

### Objetivo principal

> Caracterizar subpoblaciones de mujeres embarazadas con distintos patrones de ingesta de ácido fólico (AF) y resultados materno-infantiles, aislando el efecto de variables socioeconómicas y de salud materna.

### Estructura del notebook

```
0. Configuración e importaciones
── SEMANA 1 ──────────────────────────────────────────
1. Carga de datos y revisión de diccionarios    (T1.1)
2. Análisis Exploratorio de Datos (EDA)         (T1.2)
   2.1 Grupos de variables
   2.2 Distribuciones univariadas
   2.3 Análisis de valores faltantes
   2.4 Detección de valores atípicos
   2.5 Correlaciones
   2.6 Análisis bivariado AF – Desenlaces
3. Estrategia de aislamiento de covariables     (T1.3)
4. Diseño y resumen del pipeline TDA            (T1.4)
── SEMANA 2 ──────────────────────────────────────────
5. Preprocesamiento final para TDA              (T2.1)
6. Clustering clásico (K-Means + Ward)          (T2.2)
7. Mapper con lente UMAP sobre ingesta de AF    (T2.3)
8. Mapper con lente de salud neonatal           (T2.4)
9. Interpretación y comparación de Mappers      (T2.5)
10. Homología persistente                        (T2.6)
11. Resumen de hallazgos y próximos pasos
```


---
# SEMANA 2: Clustering Clásico y Construcción del Mapper
> **Objetivo:** Ejecutar el clustering clásico como línea base y construir los grafos Mapper con distintas
> lentes para obtener una primera versión interpretable de las subpoblaciones.

Cubre las tareas T2.1 – T2.6 del plan de trabajo.


---
## 5. Preprocesamiento Final para TDA `[T2.1]`

Se construyen dos espacios de características a partir del Dataset 2:

- **Espacio AF:** variables de ingesta cuantitativa de ácido fólico.
- **Espacio Neonatal:** variables de desenlaces materno-infantiles.

Ambos espacios se normalizan con `StandardScaler` para garantizar que todas las variables
tengan igual contribución en las métricas de distancia y en la proyección UMAP.


In [ ]:
# ── Construcción del espacio AF (Dataset 2) ──────────────────────────────
AF_FEATURES = [
    'mgAF/día SAF',                  # AF de suplementos SAF (mg/día)
    'mgAF/día MAF 1°T',              # AF de multivitamínico MAF (mg/día)
    'mg/d AF total pan',              # AF de pan (mg/día)
    'Total mg/d AF  suple y pan',     # AF total (supl + pan)
    'Período consumo SAF+OSAF',       # Período de consumo (codificado)
    '¿Consume SAF?',                  # Consumió suplemento AF (binario)
    'Consumo OSAF completo',          # Consumió otro suplemento AF
    '¿Consumió MAF?',                 # Consumió multivitamínico con AF
]

NEONATAL_FEATURES = [
    'PN hijo (g)',                     # Peso al nacer (g)
    'EG hijo (sem)',                   # Edad gestacional (semanas)
    'Dif peso mamá',                   # Delta peso gestacional (kg)
    'IMC antes',                       # IMC pregestacional
    '¿Hijo nace c/problema de salud?', # Problema de salud en RN (binario)
]

ALL_FEATURES = list(set(AF_FEATURES + NEONATAL_FEATURES))

# Dataset de trabajo: eliminar filas con faltantes en las features seleccionadas
df_tda = df2[ALL_FEATURES].copy()

# Convert all to numeric
for col in df_tda.columns:
    df_tda[col] = pd.to_numeric(df_tda[col], errors='coerce')

# Imputar faltantes numéricos con mediana
for col in df_tda.columns:
    if df_tda[col].dtype in ['float64', 'int64']:
        df_tda[col].fillna(df_tda[col].median(), inplace=True)

df_tda.dropna(inplace=True)
print(f'Dataset TDA: {df_tda.shape[0]} registros × {df_tda.shape[1]} variables')
print(f'Variables AF: {AF_FEATURES}')
print(f'Variables neonatales: {NEONATAL_FEATURES}')


In [ ]:
# ── Normalización y construcción de espacios ──────────────────────────────
scaler = StandardScaler()

# Forzar tipos numéricos
df_tda[AF_FEATURES] = df_tda[AF_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
df_tda[NEONATAL_FEATURES] = df_tda[NEONATAL_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(df_tda[NEONATAL_FEATURES].median())

# Espacio AF
X_af_raw = df_tda[AF_FEATURES].values
X_af     = scaler.fit_transform(X_af_raw)

# Espacio neonatal
X_neo_raw = df_tda[NEONATAL_FEATURES].values
X_neo     = scaler.fit_transform(X_neo_raw)

# Espacio combinado (para clustering clásico)
X_all_raw = df_tda[ALL_FEATURES].values
X_all     = StandardScaler().fit_transform(X_all_raw)

print(f'X_af (espacio AF):       {X_af.shape}')
print(f'X_neo (espacio neonatal): {X_neo.shape}')
print(f'X_all (combinado):       {X_all.shape}')
print('\nEstadísticas post-normalización (medias deben ser ~0):')
print(f'  X_af  media={X_af.mean():.4f}, std={X_af.std():.4f}')
print(f'  X_neo media={X_neo.mean():.4f}, std={X_neo.std():.4f}')


In [ ]:
# ── PCA exploratoria: ¿cuánta varianza capturan las primeras componentes? ─
pca_af  = PCA().fit(X_af)
pca_neo = PCA().fit(X_neo)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Varianza Explicada por PCA — Espacios AF y Neonatal', fontsize=12, fontweight='bold')

for ax, pca, title in zip(axes, [pca_af, pca_neo], ['Espacio AF', 'Espacio Neonatal']):
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    ax.bar(range(1, len(cum_var)+1), pca.explained_variance_ratio_,
           alpha=0.7, color='steelblue', label='Varianza individual')
    ax.plot(range(1, len(cum_var)+1), cum_var, 'o-', color='crimson',
            linewidth=2, label='Varianza acumulada')
    ax.axhline(0.8, color='gray', linestyle='--', alpha=0.7, label='80%')
    ax.set_xlabel('Componente')
    ax.set_ylabel('Proporción de varianza')
    ax.set_title(title)
    ax.legend(fontsize=8)
    n80 = np.searchsorted(cum_var, 0.8) + 1
    ax.annotate(f'{n80} comp.\npara 80%', xy=(n80, 0.8), xytext=(n80+0.5, 0.65),
                arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_pca_varianza.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


---
## 6. Clustering Clásico `[T2.2]`

Se aplican dos algoritmos de clustering sobre el espacio AF completo:

1. **K-Means:** método particional iterativo. Se determina el número óptimo de clusters
   mediante la curva del codo (inercia), el coeficiente de Silhouette y el índice de Calinski-Harabasz.
2. **Clustering jerárquico aglomerativo (Ward):** construye una jerarquía de agrupaciones
   minimizando la varianza intra-cluster. Se visualiza mediante dendrograma.

Estos clusters servirán como **línea base** para comparar con los grupos topológicos del Mapper.


In [ ]:
# ── K-Means: curva del codo y métricas de validación ─────────────────────
k_range = range(2, 11)
inertias, silhouettes, ch_scores, db_scores = [], [], [], []

for k in k_range:
    km_model = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km_model.fit_predict(X_af)
    inertias.append(km_model.inertia_)
    silhouettes.append(silhouette_score(X_af, labels))
    ch_scores.append(calinski_harabasz_score(X_af, labels))
    db_scores.append(davies_bouldin_score(X_af, labels))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('K-Means: Determinación del Número Óptimo de Clusters (Espacio AF)', fontsize=12, fontweight='bold')

for ax, vals, title, better in zip(
    axes.flat,
    [inertias, silhouettes, ch_scores, db_scores],
    ['Inercia (Elbow)', 'Silhouette Score ↑', 'Calinski-Harabasz ↑', 'Davies-Bouldin ↓'],
    [False, True, True, False]
):
    ax.plot(list(k_range), vals, 'o-', color='steelblue', linewidth=2)
    best_k = list(k_range)[np.argmax(vals) if better else np.argmin(vals)]
    ax.axvline(best_k, color='crimson', linestyle='--', alpha=0.7, label=f'k={best_k}')
    ax.set_xlabel('Número de clusters (k)')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xticks(list(k_range))

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_kmeans_validacion.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Resumen
val_df = pd.DataFrame({
    'k': list(k_range),
    'Inercia': inertias,
    'Silhouette': silhouettes,
    'Calinski-Harabasz': ch_scores,
    'Davies-Bouldin': db_scores
})
print(val_df.to_string(index=False))


In [ ]:
# ── K-Means: ajuste con k óptimo y caracterización de clusters ────────────
# El k óptimo se selecciona maximizando el índice Silhouette
k_opt = list(k_range)[np.argmax(silhouettes)]
print(f'K óptimo seleccionado (max Silhouette): k = {k_opt}')

km_final = KMeans(n_clusters=k_opt, random_state=42, n_init=15)
df_tda['cluster_kmeans'] = km_final.fit_predict(X_af)

# Caracterización estadística por cluster
char_vars = ['Total mg/d AF  suple y pan', 'mgAF/día SAF',
             'mg/d AF total pan', 'PN hijo (g)', 'EG hijo (sem)', 'IMC antes', 'Dif peso mamá']
char_labels = ['AF Total (mg/d)', 'AF SAF (mg/d)', 'AF Pan (mg/d)',
                'Peso RN (g)', 'Edad gest (sem)', 'IMC pregest', 'Δ Peso gest (kg)']

char_df = df_tda.copy()
for col in char_vars:
    if col not in char_df.columns:
        char_df[col] = pd.to_numeric(df2.loc[df_tda.index, col], errors='coerce')

print(f'\nTamaño de clusters:')
print(df_tda['cluster_kmeans'].value_counts().sort_index())

print('\nMedianas por cluster:')
resumen = df_tda.groupby('cluster_kmeans')[char_vars].median()
resumen.index.name = 'Cluster'
resumen.columns = char_labels
resumen


In [ ]:
# ── Visualización: clusters K-Means en PCA 2D ────────────────────────────
pca2d = PCA(n_components=2, random_state=42)
X_pca2d = pca2d.fit_transform(X_af)

fig, ax = plt.subplots(figsize=(9, 6))
palette = plt.cm.Set2(np.linspace(0, 1, k_opt))
for k_i in range(k_opt):
    mask = df_tda['cluster_kmeans'] == k_i
    ax.scatter(X_pca2d[mask, 0], X_pca2d[mask, 1],
               color=palette[k_i], alpha=0.6, s=25, label=f'Cluster {k_i}')

# Centroides
centroids_pca = pca2d.transform(km_final.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c='black', s=150, marker='X', zorder=5, label='Centroides')

ax.set_xlabel(f'PC1 ({pca_af.explained_variance_ratio_[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2 ({pca_af.explained_variance_ratio_[1]*100:.1f}% var.)')
ax.set_title(f'Clusters K-Means (k={k_opt}) en PCA 2D — Espacio AF', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_kmeans_pca2d.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Clustering Jerárquico Ward + Dendrograma ──────────────────────────────
linkage_matrix = linkage(X_af, method='ward', metric='euclidean')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    linkage_matrix,
    truncate_mode='level',
    p=5,
    show_leaf_counts=True,
    ax=ax,
    color_threshold=linkage_matrix[-k_opt+1, 2],
    above_threshold_color='gray'
)
ax.set_title('Dendrograma — Clustering Jerárquico Ward (Espacio AF)', fontsize=12, fontweight='bold')
ax.set_xlabel('Muestra (o grupo de muestras)')
ax.set_ylabel('Distancia Ward')
ax.axhline(linkage_matrix[-k_opt+1, 2], color='crimson', linestyle='--',
           linewidth=1.5, label=f'Corte para k={k_opt}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_dendrograma_ward.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Clustering Ward: etiquetas y acuerdo con K-Means ─────────────────────
df_tda['cluster_ward'] = fcluster(linkage_matrix, t=k_opt, criterion='maxclust') - 1

print('Tamaño de clusters Ward:')
print(df_tda['cluster_ward'].value_counts().sort_index())

# Acuerdo entre K-Means y Ward
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
ari  = adjusted_rand_score(df_tda['cluster_kmeans'], df_tda['cluster_ward'])
ami  = adjusted_mutual_info_score(df_tda['cluster_kmeans'], df_tda['cluster_ward'])
print(f'\nAjuste entre K-Means y Ward:')
print(f'  Adjusted Rand Index (ARI): {ari:.4f}  (1=acuerdo perfecto, 0=aleatorio)')
print(f'  Adjusted Mutual Info (AMI): {ami:.4f}')


In [ ]:
# ── Boxplots por cluster K-Means: variables AF y neonatales ──────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Caracterización de Clusters K-Means (k={k_opt}) — Espacio AF', fontsize=12, fontweight='bold')

plot_pairs = [
    ('Total mg/d AF  suple y pan',     'AF Total (mg/día)'),
    ('mgAF/día SAF',                    'AF Suplementos SAF (mg/día)'),
    ('mg/d AF total pan',               'AF Pan (mg/día)'),
    ('PN hijo (g)',                      'Peso RN (g)'),
    ('EG hijo (sem)',                    'Edad Gestacional (sem)'),
    ('IMC antes',                        'IMC Pregestacional'),
]

palette_c = {k_i: plt.cm.Set2(k_i / k_opt) for k_i in range(k_opt)}

for ax, (var, label) in zip(axes.flat, plot_pairs):
    data_by_cluster = [df_tda.loc[df_tda['cluster_kmeans'] == k_i, var].dropna().values
                       for k_i in range(k_opt)]
    bp = ax.boxplot(data_by_cluster, patch_artist=True,
                    medianprops=dict(color='crimson', linewidth=2))
    for patch, k_i in zip(bp['boxes'], range(k_opt)):
        patch.set_facecolor(palette_c[k_i])
        patch.set_alpha(0.75)
    ax.set_xticklabels([f'C{k_i}' for k_i in range(k_opt)])
    ax.set_title(label, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_clusters_caracterizacion.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


---
## 7. Mapper con Lente UMAP sobre Ingesta de AF `[T2.3]`

Se construye el grafo Mapper utilizando como **lente** la proyección UMAP del espacio de variables
de ingesta de AF. La lente revela la estructura topológica de los datos según su patrón de
consumo de ácido fólico, sin imponer una partición rígida como el clustering.

**Hiperparámetros Mapper:**
- `n_cubes=15`, `perc_overlap=0.4` (resolución y ganancia del Cover)
- Clustering interno: DBSCAN (adaptativo a la densidad local de cada cubo)
- Métrica de distancia: Euclidiana sobre espacio normalizado


In [ ]:
pip install umap-learn

In [ ]:
import umap
# o también: import umap.umap_ as umap

reducer_af = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

X_umap_af = reducer_af.fit_transform(X_af)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    X_umap_af[:, 0],
    X_umap_af[:, 1],
    c=df_tda['Total mg/d AF  suple y pan'].values,
    cmap='viridis',
    alpha=0.6,
    s=15
)

plt.colorbar(sc, ax=ax, label='AF Total (mg/día)')
ax.set_title('UMAP — Espacio AF (coloreado por ingesta total de AF)', fontsize=11, fontweight='bold')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_umap_af.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Construcción del Mapper con lente UMAP-AF ────────────────────────────
from sklearn.cluster import DBSCAN

mapper = km.KeplerMapper(verbose=0)

# Lente: proyección UMAP sobre espacio AF
lens_af = X_umap_af

# Grafo Mapper
graph_af = mapper.map(
    lens_af,
    X_af,
    cover=Cover(n_cubes=15, perc_overlap=0.4),
    clusterer=DBSCAN(eps=0.5, min_samples=3)
)

n_nodes_af = len(graph_af['nodes'])
n_edges_af = sum(len(v) for v in graph_af['links'].values()) // 2
print(f'Mapper (lente AF):')
print(f'  Nodos: {n_nodes_af}')
print(f'  Aristas: {n_edges_af}')


In [ ]:
# ── Visualización HTML del Mapper (lente AF) ──────────────────────────────
# Se colorea cada nodo por la mediana de AF total del grupo
node_color_af = {}
for node_id, member_ids in graph_af['nodes'].items():
    node_color_af[node_id] = float(df_tda.iloc[member_ids]['Total mg/d AF  suple y pan'].median())

# KeplerMapper 2.x requiere color_values de longitud igual al n de datos (no de nodos)
color_af_por_punto = np.zeros(X_af.shape[0])
for node_id, member_ids in graph_af['nodes'].items():
    val = node_color_af.get(node_id, 0)
    for idx in member_ids:
        color_af_por_punto[idx] = val

html_path_af = os.path.join(OUT_DIR, 'mapper_af_lens.html')
mapper.visualize(
    graph_af,
    color_values=color_af_por_punto,
    color_function_name='Mediana AF Total (mg/día)',
    title='Mapper TDA — Lente uMAP sobre Ingesta de Ácido Fólico',
    path_html=html_path_af
)
print(f'Visualización guardada en: {html_path_af}')
print('Abre el archivo HTML en tu navegador para explorar el grafo interactivo.')


In [ ]:
# ── Estadísticas por nodo del Mapper (lente AF) ───────────────────────────
node_stats_af = []
for node_id, member_ids in graph_af['nodes'].items():
    subset = df_tda.iloc[member_ids]
    node_stats_af.append({
        'Nodo': node_id,
        'N miembros': len(member_ids),
        'AF Total mediana (mg/d)': subset['Total mg/d AF  suple y pan'].median(),
        'AF SAF mediana (mg/d)': subset['mgAF/día SAF'].median(),
        'Peso RN mediana (g)': subset['PN hijo (g)'].median(),
        'Edad gest mediana (sem)': subset['EG hijo (sem)'].median(),
        '% prob. salud RN': (subset['¿Hijo nace c/problema de salud?'] == 1).mean() * 100
    })

node_df_af = pd.DataFrame(node_stats_af).sort_values('AF Total mediana (mg/d)', ascending=False)
print(f'Estadísticas por nodo (Mapper lente AF) — {len(node_df_af)} nodos:')
node_df_af.head(10)


In [ ]:
!pip install networkx

In [ ]:
# ── Visualización estática del grafo Mapper (lente AF) ────────────────────
# Se usa networkx si está disponible, si no se grafica con scatter en UMAP space
try:
    import networkx as nx

    G_af = nx.Graph()
    node_sizes = {n: len(v) for n, v in graph_af['nodes'].items()}
    node_colors_plot = {n: node_color_af.get(n, 0) for n in graph_af['nodes']}

    for node in graph_af['nodes']:
        G_af.add_node(node)
    for node, neighbors in graph_af['links'].items():
        for nb in neighbors:
            G_af.add_edge(node, nb)

    pos = nx.spring_layout(G_af, seed=42, k=0.5)
    fig, ax = plt.subplots(figsize=(11, 8))

    node_list = list(G_af.nodes())
    sizes  = [node_sizes.get(n, 1) * 15 for n in node_list]
    colors = [node_colors_plot.get(n, 0) for n in node_list]

    nx.draw_networkx_edges(G_af, pos, ax=ax, alpha=0.3, edge_color='gray')
    sc = nx.draw_networkx_nodes(G_af, pos, ax=ax, nodelist=node_list,
                                 node_size=sizes, node_color=colors,
                                 cmap='viridis', alpha=0.85)
    plt.colorbar(plt.cm.ScalarMappable(cmap='viridis',
                 norm=plt.Normalize(vmin=min(colors), vmax=max(colors))),
                 ax=ax, label='Mediana AF Total (mg/día)')
    ax.set_title('Grafo Mapper — Lente UMAP sobre Ingesta de AF\n'
                 '(Tamaño de nodo = N miembros; Color = AF Total)', fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'S2_mapper_af_grafo.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
except ImportError:
    print('networkx no instalado. Instalar con: pip install networkx')
    print('La visualización interactiva en HTML sigue disponible.')


---
## 8. Mapper con Lente de Salud Neonatal `[T2.4]`

Se construye un segundo grafo Mapper usando como **lente** la proyección UMAP del espacio
de **desenlaces neonatales** (peso al nacer, edad gestacional, delta peso gestacional, IMC
pregestacional, problema de salud en RN).

Este segundo Mapper captura la estructura de los datos según los **resultados de salud**,
y su comparación con el Mapper de AF permite identificar si los grupos con patrones similares
de AF también comparten resultados similares, o si la estructura difiere.


In [ ]:
# ── UMAP sobre espacio neonatal ───────────────────────────────────────────

reducer_neo = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

X_umap_neo = reducer_neo.fit_transform(X_neo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coloreado por peso RN
sc1 = axes[0].scatter(
    X_umap_neo[:, 0],
    X_umap_neo[:, 1],
    c=df_tda['PN hijo (g)'].values,
    cmap='plasma',
    alpha=0.6,
    s=15
)

plt.colorbar(sc1, ax=axes[0], label='Peso RN (g)')
axes[0].set_title('UMAP Neonatal — coloreado por Peso RN', fontsize=10, fontweight='bold')

# Coloreado por problema de salud RN
colors_rn = df_tda['¿Hijo nace c/problema de salud?'].map({0: 'steelblue', 1: 'crimson', 2: 'steelblue'})

axes[1].scatter(
    X_umap_neo[:, 0],
    X_umap_neo[:, 1],
    c=colors_rn,
    alpha=0.6,
    s=15
)

from matplotlib.lines import Line2D

legend_e = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='steelblue', label='Sin problema RN', ms=9),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='crimson', label='Con problema RN', ms=9)
]

axes[1].legend(handles=legend_e, fontsize=9)
axes[1].set_title('UMAP Neonatal — coloreado por Problema Salud RN', fontsize=10, fontweight='bold')

for ax in axes:
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_umap_neonatal.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Construcción del Mapper con lente neonatal ────────────────────────────
lens_neo = X_umap_neo

graph_neo = mapper.map(
    lens_neo,
    X_neo,
    cover=Cover(n_cubes=15, perc_overlap=0.4),
    clusterer=DBSCAN(eps=0.5, min_samples=3)
)

n_nodes_neo = len(graph_neo['nodes'])
n_edges_neo = sum(len(v) for v in graph_neo['links'].values()) // 2
print(f'Mapper (lente neonatal):')
print(f'  Nodos: {n_nodes_neo}')
print(f'  Aristas: {n_edges_neo}')

# Color por mediana de peso RN
node_color_neo = {}
for node_id, member_ids in graph_neo['nodes'].items():
    node_color_neo[node_id] = float(df_tda.iloc[member_ids]['PN hijo (g)'].median())

# KeplerMapper 2.x requiere color_values de longitud igual al n de datos
color_neo_por_punto = np.zeros(X_neo.shape[0])
for node_id, member_ids in graph_neo['nodes'].items():
    val = node_color_neo.get(node_id, 0)
    for idx in member_ids:
        color_neo_por_punto[idx] = val

html_path_neo = os.path.join(OUT_DIR, 'mapper_neonatal_lens.html')
mapper.visualize(
    graph_neo,
    color_values=color_neo_por_punto,
    color_function_name='Mediana Peso RN (g)',
    title='Mapper TDA — Lente uMAP sobre Salud Neonatal',
    path_html=html_path_neo
)
print(f'Visualización guardada en: {html_path_neo}')


In [ ]:
# ── Grafo Mapper neonatal: visualización estática ─────────────────────────
try:
    import networkx as nx

    G_neo = nx.Graph()
    node_sizes_neo = {n: len(v) for n, v in graph_neo['nodes'].items()}

    for node in graph_neo['nodes']:
        G_neo.add_node(node)
    for node, neighbors in graph_neo['links'].items():
        for nb in neighbors:
            G_neo.add_edge(node, nb)

    pos_neo = nx.spring_layout(G_neo, seed=42, k=0.5)
    fig, ax = plt.subplots(figsize=(11, 8))

    node_list_neo = list(G_neo.nodes())
    sizes_neo  = [node_sizes_neo.get(n, 1) * 15 for n in node_list_neo]
    colors_neo = [node_color_neo.get(n, 0) for n in node_list_neo]

    nx.draw_networkx_edges(G_neo, pos_neo, ax=ax, alpha=0.3, edge_color='gray')
    nx.draw_networkx_nodes(G_neo, pos_neo, ax=ax, nodelist=node_list_neo,
                            node_size=sizes_neo, node_color=colors_neo,
                            cmap='plasma', alpha=0.85)
    plt.colorbar(plt.cm.ScalarMappable(cmap='plasma',
                 norm=plt.Normalize(vmin=min(colors_neo), vmax=max(colors_neo))),
                 ax=ax, label='Mediana Peso RN (g)')
    ax.set_title('Grafo Mapper — Lente UMAP sobre Desenlaces Neonatales\n'
                 '(Tamaño de nodo = N miembros; Color = Peso RN)', fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'S2_mapper_neonatal_grafo.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
except ImportError:
    print('networkx no instalado.')


---
## 9. Interpretación y Comparación de Mappers `[T2.5]`

Se identifican los nodos de interés en cada Mapper y se comparan las estructuras
generadas por ambas lentes. Un nodo con alta concentración de un desenlace adverso
y un patrón específico de AF constituye un candidato a subpoblación de interés clínico.


In [ ]:
# ── Comparación estructural: Mapper AF vs. Mapper Neonatal ────────────────
print('=== Comparación de Mappers ===')
print(f'Mapper AF:       {n_nodes_af} nodos, {n_edges_af} aristas')
print(f'Mapper Neonatal: {n_nodes_neo} nodos, {n_edges_neo} aristas')
print()

# Solapamiento de miembros entre nodos de ambos Mappers
# Para cada nodo del Mapper AF, calculamos qué nodo del Mapper Neonatal captura más miembros comunes
overlap_matrix = np.zeros((n_nodes_af, n_nodes_neo))
nodes_af_list  = list(graph_af['nodes'].keys())
nodes_neo_list = list(graph_neo['nodes'].keys())

for i, n_af in enumerate(nodes_af_list):
    set_af = set(graph_af['nodes'][n_af])
    for j, n_neo in enumerate(nodes_neo_list):
        set_neo = set(graph_neo['nodes'][n_neo])
        overlap_matrix[i, j] = len(set_af & set_neo) / len(set_af | set_neo) if set_af | set_neo else 0

print(f'Matriz de solapamiento (Jaccard): {overlap_matrix.shape[0]} × {overlap_matrix.shape[1]}')
print(f'Solapamiento promedio: {overlap_matrix[overlap_matrix > 0].mean():.4f}')
print(f'Solapamiento máximo:   {overlap_matrix.max():.4f}')


In [ ]:
# ── Mapa de calor de solapamiento entre nodos de ambos Mappers ────────────
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(overlap_matrix, ax=ax, cmap='YlOrRd', cbar_kws={'label': 'Índice de Jaccard'},
            xticklabels=False, yticklabels=False)
ax.set_xlabel('Nodos Mapper Neonatal', fontsize=11)
ax.set_ylabel('Nodos Mapper AF', fontsize=11)
ax.set_title('Solapamiento de Membresía entre Mappers (Índice de Jaccard)\n'
             'Celda brillante = muchos miembros compartidos entre ambos nodos', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_mapper_overlap.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Nodos de interés: alta proporción de problema de salud RN ─────────────
print('=== Nodos Mapper AF con mayor % de problema de salud en RN ===')
node_df_af_sorted = pd.DataFrame(node_stats_af).sort_values('% prob. salud RN', ascending=False)
print(node_df_af_sorted.head(8).to_string(index=False))

print()
print('=== Nodos con muy alta ingesta AF total ===')
node_df_af_sorted2 = pd.DataFrame(node_stats_af).sort_values('AF Total mediana (mg/d)', ascending=False)
print(node_df_af_sorted2.head(5).to_string(index=False))


---
## 10. Homología Persistente `[T2.6]`

La homología persistente captura las **características topológicas** de la nube de puntos:

- **β₀ (H₀):** Componentes conectadas — grupos de puntos separados.
- **β₁ (H₁):** Ciclos o bucles — estructuras circulares en los datos.

Se calcula el complejo de Rips sobre el espacio AF usando `ripser`, y se visualiza con
**diagramas de persistencia** (birth vs. death) y **barcodes**. Los rasgos con mayor
persistencia (barra larga) son los más robustos topológicamente.


In [ ]:

def plot_persistence(diagrams, ax, title='Diagrama de Persistencia', colors=None):
    """Dibuja diagrama de persistencia sin dependencia de persim/DejaVu."""
    if colors is None:
        colors = ['steelblue', 'crimson', 'green']
    max_val = 0
    for dim, dgm in enumerate(diagrams):
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite) > 0:
            max_val = max(max_val, finite.max())
    ax.plot([0, max_val * 1.1], [0, max_val * 1.1], 'k--', alpha=0.3, linewidth=1)
    for dim, dgm in enumerate(diagrams):
        c = colors[dim % len(colors)]
        label = f'H{dim}'
        finite = dgm[np.isfinite(dgm[:, 1])]
        inf_pts = dgm[~np.isfinite(dgm[:, 1])]
        if len(finite):
            ax.scatter(finite[:, 0], finite[:, 1], c=c, s=20, alpha=0.8, label=label)
        if len(inf_pts):
            ax.scatter(inf_pts[:, 0], [max_val * 1.08] * len(inf_pts),
                       c=c, s=40, marker='^', alpha=0.8)
    ax.set_xlabel('Nacimiento (filtración)')
    ax.set_ylabel('Muerte (filtración)')
    ax.set_title(title)
    ax.legend(fontsize=9)

# ── Homología persistente: complejo de Rips sobre espacio AF ─────────────
# Se usa una muestra (máx 300 puntos) para controlar el tiempo de cómputo
MAX_POINTS = min(300, X_af.shape[0])
np.random.seed(42)
sample_idx = np.random.choice(X_af.shape[0], MAX_POINTS, replace=False)
X_af_sample = X_af[sample_idx]

print(f'Calculando homología persistente sobre {MAX_POINTS} puntos (espacio AF)...')
result_af = ripser(X_af_sample, maxdim=1, metric='euclidean')
diagrams_af = result_af['dgms']

print(f'H₀ (componentes): {len(diagrams_af[0])} rasgos')
print(f'H₁ (ciclos):      {len(diagrams_af[1])} rasgos')


In [ ]:
# ── Diagramas de persistencia y barcodes ─────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)
fig.suptitle('Homología Persistente — Espacio AF (Rips complex)', fontsize=13, fontweight='bold')

# Diagrama de persistencia (scatter birth vs. death)
ax_diag = fig.add_subplot(gs[:, 0])
plot_persistence(diagrams_af, ax=ax_diag, title='Diagrama de Persistencia (H₀ y H₁)')
ax_diag.set_title('Diagrama de Persistencia (H₀ y H₁)', fontsize=11)

# Barcodes H₀
ax_h0 = fig.add_subplot(gs[0, 1])
dgm0 = diagrams_af[0]
dgm0_finite = dgm0[np.isfinite(dgm0[:, 1])]
persistence0 = dgm0_finite[:, 1] - dgm0_finite[:, 0]
sorted_idx0  = np.argsort(persistence0)[::-1]
for i, idx in enumerate(sorted_idx0[:20]):
    ax_h0.barh(i, dgm0_finite[idx, 1] - dgm0_finite[idx, 0],
               left=dgm0_finite[idx, 0], height=0.7,
               color='steelblue', alpha=0.7)
ax_h0.set_xlabel('Filtración (distancia)')
ax_h0.set_title('Barcode H₀ — Componentes conectadas (top 20)', fontsize=10)
ax_h0.set_yticks([])

# Barcodes H₁
ax_h1 = fig.add_subplot(gs[1, 1])
dgm1 = diagrams_af[1]
if len(dgm1) > 0:
    dgm1_finite = dgm1[np.isfinite(dgm1[:, 1])]
    if len(dgm1_finite) > 0:
        persistence1 = dgm1_finite[:, 1] - dgm1_finite[:, 0]
        sorted_idx1  = np.argsort(persistence1)[::-1]
        for i, idx in enumerate(sorted_idx1[:20]):
            ax_h1.barh(i, dgm1_finite[idx, 1] - dgm1_finite[idx, 0],
                       left=dgm1_finite[idx, 0], height=0.7,
                       color='crimson', alpha=0.7)
    ax_h1.set_xlabel('Filtración (distancia)')
    ax_h1.set_title('Barcode H₁ — Ciclos (top 20)', fontsize=10)
    ax_h1.set_yticks([])
else:
    ax_h1.text(0.5, 0.5, 'Sin ciclos detectados (H₁)', ha='center', va='center', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_homologia_persistente.png'), dpi=150, bbox_inches='tight')
plt.show()

plt.close()


In [ ]:
# ── Homología persistente: espacio neonatal ───────────────────────────────
# Muestreo independiente para espacio neonatal (puede tener distinto n)
neo_sample_idx = np.random.choice(X_neo.shape[0], min(MAX_POINTS, X_neo.shape[0]), replace=False)
X_neo_sample = X_neo[neo_sample_idx]
print(f'Calculando homología persistente sobre {MAX_POINTS} puntos (espacio neonatal)...')
result_neo   = ripser(X_neo_sample, maxdim=1, metric='euclidean')
diagrams_neo = result_neo['dgms']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparación de Diagramas de Persistencia: Espacio AF vs. Neonatal', fontsize=12, fontweight='bold')

plot_persistence(diagrams_af, ax=axes[0], title='Espacio AF')
axes[0].set_title('Espacio AF')
plot_persistence(diagrams_neo, ax=axes[1], title='Espacio Neonatal')
axes[1].set_title('Espacio Neonatal')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'S2_persistencia_comparacion.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Métricas de persistencia
for name, dgms in [('AF', diagrams_af), ('Neonatal', diagrams_neo)]:
    for dim in [0, 1]:
        dgm = dgms[dim]
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite) > 0:
            pers = finite[:, 1] - finite[:, 0]
            print(f'H{dim} ({name}): {len(finite)} rasgos | max persistencia={pers.max():.4f} | media={pers.mean():.4f}')


---
## 11. Resumen de Hallazgos y Próximos Pasos

### Hallazgos de Semana 1

- Los **datasets procesados** son de buena calidad. Los valores faltantes en `uf_af` y `consumio_saf_binario` (44.7%) son **MNAR** (Missing Not At Random): corresponden a no consumidoras de suplemento, lo que es información analíticamente relevante.
- La **distribución de AF total** muestra alta asimetría positiva: la mayoría de las mujeres tiene ingestas bajas, con una cola de alta ingesta por suplementación.
- La correlación de Spearman entre AF total y peso al nacer es estadísticamente significativa pero de magnitud moderada, lo que sugiere la presencia de factores confusores relevantes.
- La **regresión de aislamiento** muestra que las covariables (edad, IMC, educación, paridad) explican entre un 5-15% de la varianza de los desenlaces — la mayor parte de la variación permanece sin explicar y es el foco del análisis TDA.

### Hallazgos de Semana 2

- El **clustering clásico K-Means** identificó grupos con patrones diferenciados de ingesta AF. Los clusters con mayor suplementación muestran medianas de peso al nacer más altas, aunque la diferencia no es dramática, consistente con la correlación moderada observada.
- El **Mapper con lente UMAP-AF** genera una estructura topológica no lineal que captura más matices que K-Means: se identifican nodos con muy alta ingesta de AF vía pan (fortification) sin suplementación, comparables en resultados a nodos con suplementación activa.
- El **Mapper con lente neonatal** revela clusters con resultados adversos (bajo peso, prematurez) que no siempre corresponden a los mismos grupos del Mapper AF, sugiriendo que la ingesta de AF no es el único determinante.
- La **homología persistente** (H₀, H₁) caracteriza la topología global de ambos espacios y servirá como complemento cuantitativo a los grafos Mapper en el informe de avance.

### Próximos pasos (Semana 3 — T3.1 a T3.5)

1. **Análisis ML** sobre las subpoblaciones del Mapper: Random Forest + SHAP para identificar los predictores más relevantes de cada perfil.
2. **Comparación formal de Mappers** con métricas cuantitativas (distancia de Gromov-Hausdorff o similares).
3. **Caracterización estadística completa** de subpoblaciones con pruebas de hipótesis.
4. **Redacción del informe de avance** con todos los resultados de semanas 1 y 2.


In [ ]:
# ── Resumen de archivos generados ────────────────────────────────────────
archivos = sorted(os.listdir(OUT_DIR))
print(f'Archivos generados en outputs/ ({len(archivos)} total):')
for f in archivos:
    ruta = os.path.join(OUT_DIR, f)
    kb = os.path.getsize(ruta) / 1024
    print(f'  {f:<55} {kb:.1f} KB')
